# Making plots for Jekyll website

## Making Graph 1 with interactivity

In [2]:
import pandas as pd

# Read original big CSV
df = pd.read_csv('Electric_Vehicle_Population_Data copy.csv')

# Your existing logic for top makes/counties
top_makes = df['Make'].value_counts().head(15).index.tolist()
top_counties = df['County'].value_counts().head(20).index.tolist()

# Filter to just what you use in chart1
df_filtered_makes = df[df['Make'].isin(top_makes) & df['County'].isin(top_counties)]

# (Optional but good) Keep only columns used in your charts
cols_needed = ['Make', 'County', 'Model Year']
df_small = df_filtered_makes[cols_needed]

# Save this to a CSV INSIDE your Jekyll repo
df_small.to_csv('/Users/danielkim/Jigs1121.github.io/python_notebooks/electric_filtered.csv', index=False)

In [3]:
import altair as alt

alt.data_transformers.disable_max_rows()

data_url = 'https://raw.githubusercontent.com/Jigs1121/Jigs1121.github.io/refs/heads/main/electric_filtered.csv'

brush = alt.selection_interval(encodings=['x', 'y'])

chart1 = alt.Chart(data_url).mark_rect().encode(
    alt.X('Make:N', title='Vehicle Make', sort='-y'),
    alt.Y('County:N', title='County', sort='-x'),
    alt.Color('count():Q', 
              scale=alt.Scale(scheme='blues'), 
              title='Vehicle Count'),
    tooltip=['Make:N', 'County:N', alt.Tooltip('count():Q', title='Count')]
).properties(
    width=500,
    height=600,
    title='Electric Vehicle Registrations by County and Make (Brush to Filter)'
).add_params(
    brush
)

chart2 = alt.Chart(data_url).mark_bar().encode(
    alt.X('count():Q', title='Number of Vehicles'),
    alt.Y('Model Year:O', 
          title='Model Year',
          sort='descending'),
    alt.Color('Model Year:O', 
              scale=alt.Scale(scheme='viridis'),
              legend=None),
    tooltip=[alt.Tooltip('Model Year:O'), alt.Tooltip('count():Q', title='Count')]
).transform_filter(
    brush
).properties(
    width=400,
    height=600,
    title='Model Year Distribution (Updates with Brush Selection)'
)

dashboard = chart1 | chart2
dashboard

alt.HConcatChart(...)

In [4]:
myJekyllDir = '/Users/danielkim/Jigs1121.github.io/assets/json/'
dashboard.save(myJekyllDir + 'electric.json')